In [1]:
# Per l'esecuzione su CPU
!pip install onnxruntime

   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
    --------------------------------------- 0.3/12.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.9 MB 1.2 MB/s eta 0:00:11
   -- ------------------------------------- 0.8/12.9 MB 931.2 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/12.9 MB 1.0 MB/s eta 0:00:12
   ---- ----------------------------------- 1.3/12.9 MB 1.2 MB/s eta 0:00:10
   ---- ----------------------------------- 1.3/12.9 MB 1.2 MB/s eta 0:00:10
   ---- ----------------------------------- 1.6/12.9 MB 1.1 MB/s eta 0:00:11
   ------ --------------------------------- 2.1/12.9 MB 1.2 MB/s eta 0:00:10
   -------- ------------------------------- 2.6/12.9 MB 1.3 MB/s eta 0:00:08
   -------- ------------------------------- 2.9/12.9 MB 1.3 MB/s eta 0:00:08
   --------- ------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import onnxruntime as ort
import numpy as np
import time

In [31]:
# 1. Carica il modello ONNX
# Se hai installato onnxruntime-gpu, cercherà prima di usare la GPU, altrimenti userà la CPU.
onnx_file_path = "./jamming_model.onnx"
sess = ort.InferenceSession(onnx_file_path)

# Estrai dinamicamente i nomi degli input (dovrebbero essere 'spectrogram' e 'features')
input_name_spec = sess.get_inputs()[0].name
input_name_feat = sess.get_inputs()[1].name

print(f"Modello caricato. Input attesi: {input_name_spec}, {input_name_feat}")
print("Avvio della simulazione real-time (premi Ctrl+C per interrompere)...\n")

# Dimensioni ipotetiche dello spettrogramma (sostituiscile con le tue dimensioni reali, es. 128x128)
H, W = 128, 873 
idx = 0
# read csv file with features
import pandas as pd
fake_features_total = pd.read_csv("./features.csv").values.astype(np.float32)

# apro json chiamato class_names e leggo le classi in una lista che uso per la pirnt della classe predetta
import json
import os

with open("./class_names.json", "r") as f:
    class_names = json.load(f)

spec_path = "./tick_p7_samples/"

try:
    for filename in sorted(os.listdir(spec_path)):
        if filename.endswith('.npy'):
            if idx >= 10:  # Simula 10 iterazioni, poi ricomincia da 0
                idx = 0
            # --- A. FASE DI ACQUISIZIONE (Simulata) ---

            # carico spettrogrammi da cartella tick_p7_samples già dimensionati correttamente
            fake_spectrogram = np.load(f"{spec_path}{filename}", allow_pickle=True).astype(np.float32)

            # devo aggiungere due dimansioni a fake spectrogram per adattarlo alla forma (1, 1, H, W) richiesta dal modello
            fake_spectrogram = np.expand_dims(fake_spectrogram, axis=0)  # Aggiunge la dimensione del batch
            fake_spectrogram = np.expand_dims(fake_spectrogram, axis=0)  # Aggiunge la dimensione del canale

            # carico features da cartella tick_p7_samples, leggi le prime 16 colonne (le altre sono label o altre info)
            fake_features = fake_features_total[idx, :16]  # Seleziona solo le prime 16 colonne

            #devo aggiungere una dimensione a fake features per adattarlo alla forma (1, 16) richiesta dal modello
            fake_features = np.expand_dims(fake_features, axis=0)  # Aggiunge la dimensione del batch

            #print(fake_spectrogram.shape)
            #print(fake_features.shape)

            inputs = {
                input_name_spec: fake_spectrogram,
                input_name_feat: fake_features
            }

            # --- B. FASE DI INFERENZA ---
            start_time = time.perf_counter()
            
            # Esegue il modello. Passare 'None' come primo argomento fa restituire tutti gli output
            outputs = sess.run(None, inputs)
            
            # Estrai gli output (logits, penultimate, energy)
            logits = outputs[0]
            energy = outputs[2]
            penultimate = outputs[1]

            
            end_time = time.perf_counter()

            # --- C. CALCOLO PERFORMANCE E RISULTATI ---
            inference_time_ms = (end_time - start_time) * 1000
            fps = 1.0 / (end_time - start_time)

            
            # Trova la classe predetta (l'indice con il valore massimo nei logits)
            predicted_class = np.argmax(logits[0])
            class_predicted_name = class_names[predicted_class] if predicted_class < len(class_names) else "Unknown"
            print(f"Classe predetta: {class_predicted_name} | Tempo: {inference_time_ms:.2f} ms | FPS: {fps:.1f}")
            print(f"Logits: {logits}")
            print(f"Energy: {energy}")
            print (".........")
            #print(penultimate)
            print(penultimate.shape)
            
            # Opzionale: inserisci un piccolo ritardo per simulare il rateo di arrivo dei dati reali (es. 50ms)
            time.sleep(0.05)
            idx += 1

except KeyboardInterrupt:
    print("\nSimulazione real-time fermata dall'utente.")

Modello caricato. Input attesi: spectrogram, features
Avvio della simulazione real-time (premi Ctrl+C per interrompere)...

Classe predetta: LN_LOW | Tempo: 17.03 ms | FPS: 58.7
Logits: [[-11.86938      9.898248    -5.756039   -14.687952    -0.10104371
   -9.26422    -14.585954    -3.4977229  -11.245522   -15.290658
   -9.995787   -10.306123   -14.850632    -2.5900257   -7.923611
  -13.723749  ]]
Energy: [9.898298]
.........
(1, 528)
Classe predetta: TRI_MID | Tempo: 15.96 ms | FPS: 62.6
Logits: [[-10.330211  -11.403255   -8.583575  -13.21978    -9.583612  -11.614791
  -14.115413  -13.093326  -10.040231  -11.268368  -13.730929   -7.2393436
   -9.908993    3.9710233  14.141552    2.5252185]]
Energy: [14.1416]
.........
(1, 528)
Classe predetta: TRI_MID | Tempo: 8.44 ms | FPS: 118.5
Logits: [[-10.463902  -11.541885   -8.693071  -13.382528   -9.685653  -11.764247
  -14.32173   -13.306958  -10.167274  -11.401326  -13.944413   -7.3077197
  -10.023401    4.0167694  14.331142    2.602655 ]]
E